### Setup 

In [1]:
# Import libraries
import os
import json
import requests
import io
import boto3
import pandas as pd
from dotenv import load_dotenv

In [2]:
# Load urls, passwords and access keys
load_dotenv()
API_HOST = os.getenv("API_HOST")
ACCESS_KEY = os.getenv("AWS_KEY")
ACCESS_SECRET = os.getenv("AWS_SECRET")
upload_bucket = os.getenv("S3_BUCKET")
filepath = os.getenv("FILE")

### Extract data from data.cms.gov API

In [3]:
# Get URL
url = f"{API_HOST}"
#
# Define extract function
def extract_data():
      try:
            response = requests.get(url, params=params)
            response.raise_for_status() # raises an exception for http errors
            return response.json()
      except requests.exceptions.RequestException as err:
            print(f"An error occured: {err}")
#                    
rows = []
offset = 0
limit = 1500  
#
print("Connecting to CMS API")
# API is paginated - fetch all pages with a while loop         
while True:
      params = {
      "limit": limit,
      "offset": offset
      }    
      result = extract_data() # get data from API
      data = result["results"] # API response is a dictionnary of lists - we are only keeping the results list
      if not data:
            break
      rows.extend(data)
      offset += limit
if len(rows) > 0:
      print ("CMS API data received")
      print(f"Downloaded {len(rows)} rows sucessfully") # print number of rows downloaded

Connecting to CMS API
CMS API data received
Downloaded 325720 rows sucessfully


### Create dataframe

In [4]:
# Parse API response into pandas dataframe
df = pd.DataFrame(rows)


### Load data in AWS S3

In [ ]:
# Define load function
def load():
        try:
                print("Importing data in S3") 
                #
                s3_client = boto3.client('s3', aws_access_key_id=ACCESS_KEY, aws_secret_access_key=ACCESS_SECRET, region_name='us-east-1')
                with io.StringIO() as csv_buffer:
                        df.to_csv(csv_buffer, index=False)
                        response=s3_client.put_object(
                        Bucket=upload_bucket, Key=f"raw/{filepath}", Body=csv_buffer.getvalue()
                        )
                status = response.get("ResponseMetadata", {}).get("HTTPStatusCode")
                #
                if status == 200:
                        print("Succesful S3 put_object response")
                else:
                        print(f"Unsuccesful S3 put_object response. Status - {status}")
        except Exception as e:
                print(f"Data load error: {e}")


load()



importing data in S3
Succesful S3 put_object response
